In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, math, re, time, sys, subprocess, pkgutil, traceback, html
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from peft import PeftModel
from transformers import AutoTokenizer, AutoConfig, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

if pkgutil.find_loader("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44.0"])
import gradio as gr

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/B8_light_best_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
with open(os.path.join(MODEL_DIR, "export_meta.json"), "r", encoding="utf-8") as f:
    META = json.load(f)

MODEL_NAME = META["model_name"]
MAX_LENGTH = META.get("max_length", 1536)
HEAD_DROPOUT = META.get("head_dropout", 0.1)

TR_FEATURE_COLS = META["tr_feature_cols"]
CC_FEATURE_COLS = META["cc_feature_cols"]
LR_FEATURE_COLS = META["lr_feature_cols"]
GRA_FEATURE_COLS = META["gra_feature_cols"]

tr_feat_mean, tr_feat_std = np.array(META["tr_feat_mean"], dtype=np.float32), np.array(META["tr_feat_std"], dtype=np.float32)
cc_feat_mean, cc_feat_std = np.array(META["cc_feat_mean"], dtype=np.float32), np.array(META["cc_feat_std"], dtype=np.float32)
lr_feat_mean, lr_feat_std = np.array(META["lr_feat_mean"], dtype=np.float32), np.array(META["lr_feat_std"], dtype=np.float32)
gra_feat_mean, gra_feat_std = np.array(META["gra_feat_mean"], dtype=np.float32), np.array(META["gra_feat_std"], dtype=np.float32)

In [ ]:
STOPWORDS = {
    "a","an","the","and","or","but","if","while","is","am","are","was","were",
    "be","been","being","of","to","in","on","for","with","as","at","by","from",
    "that","this","these","those","it","its","he","she","they","them","their",
    "we","our","you","your","i","me","my","mine","his","her","hers","do","does",
    "did","have","has","had","will","would","can","could","should","may","might",
    "not","so","than","then","there","here","about","into","over","after","before",
    "more","most","some","any","such","no","nor","too","very"
}

DISCOURSE_MARKERS = [
    "however", "therefore", "moreover", "furthermore", "in addition",
    "for example", "for instance", "on the one hand", "on the other hand",
    "in conclusion", "to conclude", "in summary", "as a result",
    "firstly", "secondly", "finally", "besides", "nevertheless",
    "thus", "overall", "in contrast", "for this reason"
]

LONG_WORD_MIN_LEN = 7

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def normalize_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_words(text):
    text = normalize_text(text)
    return re.findall(r"[a-zA-Z']+", text)

def split_sentences(text):
    text = safe_text(text).strip()
    if not text:
        return []
    sents = re.split(r'(?<=[.!?])\s+', text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents

def split_paragraphs(text):
    text = safe_text(text)
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paras) == 0 and text.strip():
        paras = [text.strip()]
    return paras

def word_count(text):
    return len(tokenize_words(text))

def sentence_count(text):
    return len(split_sentences(text))

def unique_ratio(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / len(words)

def root_ttr(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / math.sqrt(len(words))

def repetition_ratio(words):
    if len(words) <= 1:
        return 0.0
    c = Counter(words)
    repeated = sum(v for v in c.values() if v > 1)
    return repeated / len(words)

def repeated_word_ratio(words):
    if len(words) <= 1:
        return 0.0
    repeated_pairs = 0
    for i in range(1, len(words)):
        if words[i] == words[i - 1]:
            repeated_pairs += 1
    return repeated_pairs / len(words)

def avg_word_len(words):
    if len(words) == 0:
        return 0.0
    return float(np.mean([len(w) for w in words]))

def lexical_density_proxy(words):
    if len(words) == 0:
        return 0.0
    content_like = [w for w in words if len(w) > 3 and w not in STOPWORDS]
    return len(content_like) / len(words)

def long_word_ratio(words, min_len=LONG_WORD_MIN_LEN):
    if len(words) == 0:
        return 0.0
    return sum(len(w) >= min_len for w in words) / len(words)

def sentence_lengths(sentences):
    return [len(tokenize_words(s)) for s in sentences if len(tokenize_words(s)) > 0]

def count_discourse_markers(text):
    low = normalize_text(text)
    counts = []
    found = 0
    found_types = 0
    for m in DISCOURSE_MARKERS:
        c = low.count(m)
        counts.append(c)
        if c > 0:
            found += c
            found_types += 1
    return found, found_types

def prompt_keywords(prompt):
    words = tokenize_words(prompt)
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return set(words)

def jaccard_coverage(prompt, essay):
    pk = prompt_keywords(prompt)
    ew = set(tokenize_words(essay))
    if len(pk) == 0:
        return 0.0
    return len(pk & ew) / len(pk)

def prompt_essay_similarity(prompt, essay):
    pw = prompt_keywords(prompt)
    ew = set([w for w in tokenize_words(essay) if w not in STOPWORDS and len(w) > 2])
    if len(pw) == 0 or len(ew) == 0:
        return 0.0
    return len(pw & ew) / math.sqrt(len(pw) * len(ew))

def has_opinion_statement(text):
    low = normalize_text(text)
    patterns = [
        "i believe", "i think", "in my opinion", "from my perspective",
        "personally", "it seems to me", "i would argue"
    ]
    return any(p in low for p in patterns)

def has_both_views(text):
    low = normalize_text(text)
    return (
        ("on the one hand" in low and "on the other hand" in low) or
        ("while some people think" in low and "others believe" in low) or
        ("some people argue" in low and "others believe" in low)
    )

def has_example(text):
    low = normalize_text(text)
    return any(p in low for p in ["for example", "for instance", "such as", "to illustrate"])

def has_conclusion(text):
    low = normalize_text(text)
    return any(p in low for p in ["in conclusion", "to conclude", "to sum up", "overall"])

def punct_density(text):
    words = tokenize_words(text)
    if len(words) == 0:
        return 0.0
    puncts = re.findall(r"[.,;:!?]", safe_text(text))
    return len(puncts) / len(words)

def repeated_punct_ratio(text):
    raw = safe_text(text)
    if len(raw) == 0:
        return 0.0
    repeated = re.findall(r"([!?.,;:])\1+", raw)
    return len(repeated) / len(raw)

def lowercase_sentence_start_ratio(text):
    sents = split_sentences(text)
    if len(sents) == 0:
        return 0.0
    bad = 0
    checked = 0
    for s in sents:
        m = re.search(r"[A-Za-z]", s)
        if m:
            checked += 1
            ch = s[m.start()]
            if ch.islower():
                bad += 1
    if checked == 0:
        return 0.0
    return bad / checked

def lowercase_i_ratio(text):
    toks = re.findall(r"\b[a-zA-Z']+\b", safe_text(text))
    if len(toks) == 0:
        return 0.0
    bad = sum(1 for t in toks if t == "i")
    return bad / len(toks)

def missing_terminal_punct(text):
    txt = safe_text(text).strip()
    if not txt:
        return 1.0
    return 0.0 if txt[-1] in ".!?" else 1.0

def extract_tr_features(prompt, essay):
    return {
        "tr_prompt_essay_sim": float(prompt_essay_similarity(prompt, essay)),
        "tr_prompt_keyword_coverage": float(jaccard_coverage(prompt, essay)),
        "tr_has_opinion": float(has_opinion_statement(essay)),
        "tr_has_both_views": float(has_both_views(essay)),
        "tr_has_example": float(has_example(essay)),
        "tr_has_conclusion": float(has_conclusion(essay)),
        "tr_word_count": float(word_count(essay)),
    }

def extract_cc_features(prompt, essay):
    paras = split_paragraphs(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)
    dm_count, dm_div = count_discourse_markers(essay)

    para_lens = [word_count(p) for p in paras] if len(paras) > 0 else [0]

    return {
        "cc_num_paragraphs": float(len(paras)),
        "cc_avg_paragraph_len": float(np.mean(para_lens)) if len(para_lens) > 0 else 0.0,
        "cc_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_sentence_len_std": float(np.std(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_discourse_marker_count": float(dm_count),
        "cc_discourse_marker_diversity": float(dm_div),
    }

def extract_lr_features(prompt, essay):
    words = tokenize_words(essay)

    return {
        "lr_root_ttr": float(root_ttr(words)),
        "lr_avg_word_len": float(avg_word_len(words)),
        "lr_long_word_ratio": float(long_word_ratio(words)),
        "lr_repetition_ratio": float(repetition_ratio(words)),
        "lr_unique_word_ratio": float(unique_ratio(words)),
        "lr_lexical_density_proxy": float(lexical_density_proxy(words)),
    }

def extract_gra_features(prompt, essay):
    words = tokenize_words(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)

    return {
        "gf_word_count": float(len(words)),
        "gf_sentence_count": float(len(sents)),
        "gf_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_short_sentence_ratio": float(sum(l < 8 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_long_sentence_ratio": float(sum(l > 30 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_punct_density": float(punct_density(essay)),
        "gf_repeated_punct_ratio": float(repeated_punct_ratio(essay)),
        "gf_lowercase_sent_start_ratio": float(lowercase_sentence_start_ratio(essay)),
        "gf_lowercase_i_ratio": float(lowercase_i_ratio(essay)),
        "gf_repeated_word_ratio": float(repeated_word_ratio(words)),
        "gf_missing_terminal_punct": float(missing_terminal_punct(essay)),
    }

def build_input_text(prompt, essay):
    prompt = str(prompt).strip()
    essay = str(essay).strip()
    return (
        "You are an IELTS Writing Task 2 examiner.\n"
        "Evaluate the essay based on Task Response, Coherence and Cohesion, "
        "Lexical Resource, and Grammatical Range and Accuracy.\n\n"
        "TASK PROMPT:\n"
        f"{prompt}\n\n"
        "ESSAY:\n"
        f"{essay}"
    )

def standardize_features(feat_dict, cols, mean_, std_):
    arr = np.array([feat_dict[c] for c in cols], dtype=np.float32)
    std_ = np.where(std_ < 1e-6, 1.0, std_)
    arr = (arr - mean_) / std_
    return arr

def round_to_half(x):
    return round(float(x) * 2) / 2

In [ ]:
class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, head_dropout: float = 0.1, fusion_dim: int = 256):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        base_model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        base_model.config.pad_token_id = tokenizer.pad_token_id
        base_model.config.use_cache = False

        self.backbone = PeftModel.from_pretrained(base_model, MODEL_DIR)

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)
        self.fusion_dim = fusion_dim

        self.tr_feat_dim = len(TR_FEATURE_COLS)
        self.cc_feat_dim = len(CC_FEATURE_COLS)
        self.lr_feat_dim = len(LR_FEATURE_COLS)
        self.gra_feat_dim = len(GRA_FEATURE_COLS)

        def make_branch(feat_dim):
            return nn.ModuleDict({
                "llm_proj": nn.Sequential(
                    nn.Linear(hidden_size, fusion_dim),
                    nn.GELU(),
                    nn.Dropout(head_dropout),
                ),
                "feat_proj": nn.Sequential(
                    nn.Linear(feat_dim, fusion_dim),
                    nn.GELU(),
                    nn.Dropout(head_dropout),
                ),
                "gate": nn.Sequential(
                    nn.Linear(fusion_dim * 2, fusion_dim),
                    nn.GELU(),
                    nn.Dropout(head_dropout),
                    nn.Linear(fusion_dim, 1),
                ),
                "out": nn.Sequential(
                    nn.Linear(fusion_dim, fusion_dim // 2),
                    nn.GELU(),
                    nn.Dropout(head_dropout),
                    nn.Linear(fusion_dim // 2, 1),
                )
            })

        self.tr_branch = make_branch(self.tr_feat_dim)
        self.cc_branch = make_branch(self.cc_feat_dim)
        self.lr_branch = make_branch(self.lr_feat_dim)
        self.gra_branch = make_branch(self.gra_feat_dim)

    def _mean_pool(self, hidden_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        summed = torch.sum(hidden_states * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / denom

    def _fuse_features(self, pooled, feat, branch):
        llm_emb = branch["llm_proj"](pooled)   # [B, fusion_dim]
        feat_emb = branch["feat_proj"](feat)   # [B, fusion_dim]

        gate_inp = torch.cat([llm_emb, feat_emb], dim=1)
        gate = torch.sigmoid(branch["gate"](gate_inp))   # [B, 1]

        fused_emb = gate * llm_emb + (1.0 - gate) * feat_emb
        score = branch["out"](fused_emb)  # [B, 1]
        return score, gate

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        tr_features=None,
        cc_features=None,
        lr_features=None,
        gra_features=None,
        labels=None,
        **kwargs
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        hidden_states = outputs.last_hidden_state
        pooled = self._mean_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        head_dtype = self.tr_branch["llm_proj"][0].weight.dtype
        pooled = pooled.to(dtype=head_dtype)

        tr_features = tr_features.to(device=pooled.device, dtype=head_dtype)
        cc_features = cc_features.to(device=pooled.device, dtype=head_dtype)
        lr_features = lr_features.to(device=pooled.device, dtype=head_dtype)
        gra_features = gra_features.to(device=pooled.device, dtype=head_dtype)

        tr_score, tr_gate = self._fuse_features(pooled, tr_features, self.tr_branch)
        cc_score, cc_gate = self._fuse_features(pooled, cc_features, self.cc_branch)
        lr_score, lr_gate = self._fuse_features(pooled, lr_features, self.lr_branch)
        gra_score, gra_gate = self._fuse_features(pooled, gra_features, self.gra_branch)

        logits = torch.cat([tr_score, cc_score, lr_score, gra_score], dim=1)

        return {
            "logits": logits,
            "tr_gate": tr_gate,
            "cc_gate": cc_gate,
            "lr_gate": lr_gate,
            "gra_gate": gra_gate,
        }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

fusion_dim = META.get("fusion_dim", 256)

model = QwenForIELTSMultiTask(
    MODEL_NAME,
    tokenizer,
    head_dropout=HEAD_DROPOUT,
    fusion_dim=fusion_dim,
)

head_state = torch.load(os.path.join(MODEL_DIR, "custom_heads.pt"), map_location="cpu")

print("Loading custom heads...")
print("Saved arch:", head_state.get("model_arch", {}))

model.tr_branch.load_state_dict(head_state["tr_branch"])
model.cc_branch.load_state_dict(head_state["cc_branch"])
model.lr_branch.load_state_dict(head_state["lr_branch"])
model.gra_branch.load_state_dict(head_state["gra_branch"])

model.to(DEVICE)
model.eval()

@torch.no_grad()
def predict_ielts(prompt: str, essay: str):
    tr_feat = extract_tr_features(prompt, essay)
    cc_feat = extract_cc_features(prompt, essay)
    lr_feat = extract_lr_features(prompt, essay)
    gra_feat = extract_gra_features(prompt, essay)

    tr_arr = standardize_features(tr_feat, TR_FEATURE_COLS, tr_feat_mean, tr_feat_std)
    cc_arr = standardize_features(cc_feat, CC_FEATURE_COLS, cc_feat_mean, cc_feat_std)
    lr_arr = standardize_features(lr_feat, LR_FEATURE_COLS, lr_feat_mean, lr_feat_std)
    gra_arr = standardize_features(gra_feat, GRA_FEATURE_COLS, gra_feat_mean, gra_feat_std)

    text = build_input_text(prompt, essay)
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        tr_features=torch.tensor(tr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
        cc_features=torch.tensor(cc_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
        lr_features=torch.tensor(lr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
        gra_features=torch.tensor(gra_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
    )

    preds = outputs["logits"].squeeze(0).detach().float().cpu().numpy()
    preds = np.clip(preds, 4.0, 9.0)

    raw_scores = {
        "TR": float(preds[0]),
        "CC": float(preds[1]),
        "LR": float(preds[2]),
        "GRA": float(preds[3]),
    }

    final_scores = {k: round_to_half(v) for k, v in raw_scores.items()}
    final_scores["Overall"] = round_to_half(np.mean([
        final_scores["TR"],
        final_scores["CC"],
        final_scores["LR"],
        final_scores["GRA"],
    ]))

    gate_info = {
        "tr_gate": float(outputs["tr_gate"].mean().detach().cpu()),
        "cc_gate": float(outputs["cc_gate"].mean().detach().cpu()),
        "lr_gate": float(outputs["lr_gate"].mean().detach().cpu()),
        "gra_gate": float(outputs["gra_gate"].mean().detach().cpu()),
    }

    return {
        "raw_scores": raw_scores,
        "final_scores": final_scores,
        "features": {
            "tr": tr_feat,
            "cc": cc_feat,
            "lr": lr_feat,
            "gra": gra_feat,
        },
        "gates": gate_info,
    }

def get_pred_scores(prompt, essay):
    res = predict_ielts(prompt, essay)
    return res, res["final_scores"]

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
def embed_texts(texts, batch_size=32): return embedding_model.encode(texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)

In [ ]:
def build_doc_text(row): 
    return f"[PROMPT]\n{safe_text(row['prompt'])}\n\n[ESSAY]\n{safe_text(row['essay'])}".strip()
def build_retrieval_database(csv_path):
    df = pd.read_csv(csv_path)
    df["doc_text"] = df.apply(build_doc_text, axis=1)
    return df, embed_texts(df["doc_text"].tolist())

In [ ]:
EXPLAIN_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
explain_tokenizer = AutoTokenizer.from_pretrained(EXPLAIN_MODEL_NAME, use_fast=False)
if explain_tokenizer.pad_token is None: explain_tokenizer.pad_token = explain_tokenizer.eos_token
explain_tokenizer.padding_side = "left"
explain_model = AutoModelForCausalLM.from_pretrained(EXPLAIN_MODEL_NAME, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto").eval()

In [ ]:
def retrieve_similar_essays_for_inference(prompt, essay, df, db_embeddings, top_k_vector=20, top_k_final=5, pred_result=None, pred_scores=None):
    if pred_scores is None:
        pred_result, pred_scores = get_pred_scores(prompt, essay)

    q_text = f"IELTS Writing Task 2 Prompt:\n{prompt}\n\nEssay:\n{essay}\n\nPredicted scores: TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}"
    sims = cosine_similarity(embed_texts([q_text])[0].reshape(1, -1), db_embeddings)[0]
    candidate_idx = np.argsort(-sims)[:top_k_vector]
    candidates = df.iloc[candidate_idx].copy()
    candidates["final_score"] = sims[candidate_idx]

    top_cases_df = candidates.sort_values("final_score", ascending=False).head(top_k_final)
    top_cases = top_cases_df.to_dict(orient="records")

    return {
        "pred_scores": pred_scores,
        "top_cases": top_cases
    }

In [ ]:
def mistral_explain_writer(prompt_text, temperature=0.15, max_new_tokens=600):
    full_prompt = f"[INST]\nYou are an expert IELTS examiner.\n{prompt_text}\n[/INST]"
    model_inputs = explain_tokenizer([full_prompt], return_tensors="pt", truncation=True, max_length=4096).to(explain_model.device)
    gen_kwargs = {"max_new_tokens": max_new_tokens, "repetition_penalty": 1.05}
    if temperature > 0: gen_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": 0.9})
    with torch.no_grad():
        output_ids = explain_model.generate(**model_inputs, **gen_kwargs)
    return explain_tokenizer.batch_decode(output_ids[:, model_inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def safe_json_loads(text, default=None):
    if default is None: default = {}
    match = re.search(r"\{.*\}", str(text).strip(), re.DOTALL)
    try: return json.loads(match.group(0)) if match else default
    except Exception: return default

In [ ]:
def tool_predict_scores(prompt, essay, state):
    res, scores = get_pred_scores(prompt, essay)
    state["pred_scores"] = scores
    return {"scores": scores}

tool_detect_task_mismatch

In [ ]:
def build_task_mismatch_prompt(prompt, essay, pred_scores):
    return f"""
You are a strict IELTS Writing Task 2 task-response checker.

Your job:
Determine whether the essay fully addresses the prompt, only partially addresses it, or is off-topic.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Judging rules:
1. Focus only on task alignment and relevance.
2. Do NOT judge grammar, vocabulary, or cohesion here.
3. "pass" means the essay clearly answers the prompt and stays relevant.
4. "partial_mismatch" means the essay is somewhat relevant but misses a major part of the task, misinterprets part of it, or addresses it only incompletely.
5. "off_topic" means the essay mainly answers a different question or is largely irrelevant.
6. Be conservative: do not call it off-topic unless the mismatch is clear.
7. If there is mismatch, identify the missing or misaddressed parts.

Output valid JSON only in this format:
{{
  "verdict": "pass" or "partial_mismatch" or "off_topic",
  "reason": "...",
  "missing_parts": ["...", "..."],
  "evidence": "..."
}}
""".strip()


def normalize_task_check_result(obj):
    if not isinstance(obj, dict):
        return {
            "verdict": "pass",
            "reason": "",
            "missing_parts": [],
            "evidence": "",
        }

    verdict = str(obj.get("verdict", "pass")).strip().lower()
    if verdict not in ["pass", "partial_mismatch", "off_topic"]:
        verdict = "pass"

    missing_parts = obj.get("missing_parts", [])
    if not isinstance(missing_parts, list):
        missing_parts = []

    return {
        "verdict": verdict,
        "reason": str(obj.get("reason", "")).strip(),
        "missing_parts": [str(x).strip() for x in missing_parts if str(x).strip()],
        "evidence": str(obj.get("evidence", "")).strip(),
    }


def round_to_half_band(x):
    return round(float(x) * 2) / 2


def apply_tr_guardrail(pred_scores, task_check):
    if pred_scores is None:
        return pred_scores

    guarded = dict(pred_scores)
    verdict = str(task_check.get("verdict", "pass")).strip().lower()

    old_tr = float(guarded["TR"])

    if verdict == "pass":
        new_tr = old_tr

    elif verdict == "partial_mismatch":
        # cap nhẹ cho demo
        new_tr = min(old_tr, 6.0)

    elif verdict == "off_topic":
        # cap mạnh hơn
        new_tr = min(old_tr, 5.0)

    else:
        new_tr = old_tr

    guarded["TR"] = round_to_half_band(new_tr)

    overall = np.mean([
        guarded["TR"],
        guarded["CC"],
        guarded["LR"],
        guarded["GRA"],
    ])
    guarded["Overall"] = round_to_half_band(overall)

    return guarded


def tool_detect_task_mismatch(prompt, essay, state):
    pred_scores = state["pred_scores"]

    task_prompt = build_task_mismatch_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
    )

    raw = mistral_explain_writer(
        task_prompt,
        temperature=0.0,
        max_new_tokens=500,
    )

    parsed = safe_json_loads(raw, default={})
    task_check = normalize_task_check_result(parsed)
    task_check["raw"] = str(raw)

    state["task_check"] = task_check

    original_scores = dict(state["pred_scores"])
    adjusted_scores = apply_tr_guardrail(state["pred_scores"], task_check)
    state["pred_scores"] = adjusted_scores

    return {
        "task_check": {
            "verdict": task_check["verdict"],
            "reason": task_check["reason"],
            "missing_parts": task_check["missing_parts"],
            "evidence": task_check["evidence"],
        },
        "score_adjustment": {
            "TR_before": original_scores["TR"],
            "TR_after": adjusted_scores["TR"],
            "Overall_before": original_scores["Overall"],
            "Overall_after": adjusted_scores["Overall"],
        }
    }

In [ ]:
def tool_retrieve_similar_essays(prompt, essay, df, db_embeddings, state, top_k_vector=20, top_k_final=5):
    res = retrieve_similar_essays_for_inference(prompt, essay, df, db_embeddings, top_k_vector, top_k_final, pred_scores=state.get("pred_scores"))
    state["top_cases"] = res["top_cases"]
    return {"status": f"Retrieved {len(res['top_cases'])} cases"}

Tool Generate Feedback


In [ ]:
def format_top_cases_for_prompt(top_cases, max_cases=3, max_chars_each=1200):
    if top_cases is None:
        return "[]"

    items = []
    for i, c in enumerate(top_cases[:max_cases], start=1):
        prompt_txt = str(c.get("prompt", ""))[:400]
        essay_txt = str(c.get("essay", ""))[:max_chars_each]
        tr = c.get("TR", c.get("tr", "N/A"))
        cc = c.get("CC", c.get("cc", "N/A"))
        lr = c.get("LR", c.get("lr", "N/A"))
        gra = c.get("GRA", c.get("gra", "N/A"))
        overall = c.get("Overall", c.get("overall", "N/A"))

        items.append(
            f"""
[Reference Case {i}]
Prompt: {prompt_txt}
Scores: TR={tr}, CC={cc}, LR={lr}, GRA={gra}, Overall={overall}
Essay Snippet:
{essay_txt}
""".strip()
        )
    return "\n\n".join(items) if items else "[]"


def build_generate_feedback_prompt(prompt, essay, pred_scores, top_cases):
    refs_text = format_top_cases_for_prompt(top_cases, max_cases=3, max_chars_each=900)

    return f"""
You are a strict IELTS Writing Task 2 feedback writer.

Your task:
Write criterion-level feedback for the given essay using the FIXED predicted scores below.
The feedback must be faithful to the essay itself. The retrieved reference cases are only grounding support, not text to copy.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Retrieved reference cases:
{refs_text}

Hard rules:
1. NEVER change the predicted scores.
2. Do NOT mention any alternative band score.
3. Do NOT invent weaknesses that are not visible in the essay.
4. Each criterion must stay in its own scope:
   - TR: task response, relevance, coverage, position, development
   - CC: organization, progression, paragraphing, linking
   - LR: vocabulary range, precision, repetition, collocation, word choice
   - GRA: grammar, sentence structure, articles, prepositions, agreement, verb forms, punctuation
5. Do NOT mix criteria:
   - CC must not talk about grammar or vocabulary quality
   - LR must not talk about grammar or coherence
   - GRA must not talk about idea development or vocabulary range
6. GRA comments should prefer concrete evidence from the essay when possible.
7. Keep the tone examiner-like, concise, and specific.
8. Each criterion must have exactly these three fields:
   - Strength
   - Limitation
   - Revision
9. Revision must be actionable and aligned with the criterion.
10. Do not copy wording from the retrieved reference essays.

Output valid JSON only, in exactly this format:
{{
  "TR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "CC": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "LR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "GRA": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }}
}}
""".strip()


def normalize_feedback_dict(obj):
    result = {}
    for crit in ["TR", "CC", "LR", "GRA"]:
        block = obj.get(crit, {}) if isinstance(obj, dict) else {}
        if not isinstance(block, dict):
            block = {}
        result[crit] = {
            "Strength": str(block.get("Strength", "")).strip(),
            "Limitation": str(block.get("Limitation", "")).strip(),
            "Revision": str(block.get("Revision", "")).strip(),
        }
    return result


def tool_generate_feedback(prompt, essay, state):
    pred_scores = state["pred_scores"]
    top_cases = state.get("top_cases", [])

    gen_prompt = build_generate_feedback_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        top_cases=top_cases,
    )

    raw = mistral_explain_writer(
        gen_prompt,
        temperature=0.0,
        max_new_tokens=900,
    )

    parsed = safe_json_loads(raw, default={})
    feedback = normalize_feedback_dict(parsed)

    state["feedback"] = feedback

    return {
        "feedback_generated": True,
        "criteria": list(feedback.keys()),
        "raw_preview": str(raw)[:800],
    }

Tool VerifyFeedBack

In [ ]:
def format_feedback_dict(feedback):
    if feedback is None:
        return "{}"
    try:
        return json.dumps(feedback, indent=2, ensure_ascii=False)
    except:
        return str(feedback)


def build_verify_feedback_prompt(prompt, essay, pred_scores, feedback):
    feedback_text = format_feedback_dict(feedback)

    return f"""
You are a strict IELTS Writing Task 2 feedback verifier.

Task:
Check whether the feedback is specific, evidence-based, criterion-faithful, and consistent with the FIXED predicted scores.
Do not rewrite the feedback. Only judge whether revision is needed.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Feedback to verify:
{feedback_text}

Verification principles:
1. Never change the predicted scores.
2. Judge each criterion separately: TR, CC, LR, GRA.
3. Mark "revise" only if there is a clear issue.
4. Do not over-penalize minor or debatable points.
5. Do not invent essay problems that are not visible.

Criterion-specific rules:

TR:
- Must focus on relevance, position, coverage, and development.
- Must not criticize grammar or vocabulary quality.
- Must not demand ideas outside the prompt.

CC:
- Must focus only on organization, progression, paragraphing, linking.
- Must not talk about grammar accuracy or word choice.

LR:
- Must focus only on vocabulary range, repetition, precision, collocation, wording.
- Must not treat grammar mistakes as LR problems.

GRA:
- Must focus only on grammar and sentence-level accuracy.
- Prefer concrete evidence or clearly observable patterns from the essay.
- Must not drift into idea quality or paragraph organization.

For each issue, provide:
- criterion
- problem
- reason
- fix_instruction

Output valid JSON only in this format:
{{
  "verdict": "pass" or "revise",
  "issues": [
    {{
      "criterion": "TR",
      "problem": "...",
      "reason": "...",
      "fix_instruction": "..."
    }}
  ]
}}
""".strip()


def normalize_verification_result(obj):
    if not isinstance(obj, dict):
        return {"verdict": "revise", "issues": []}

    verdict = str(obj.get("verdict", "revise")).strip().lower()
    if verdict not in ["pass", "revise"]:
        verdict = "revise"

    raw_issues = obj.get("issues", [])
    if not isinstance(raw_issues, list):
        raw_issues = []

    issues = []
    for x in raw_issues:
        if not isinstance(x, dict):
            continue
        crit = str(x.get("criterion", "")).upper().strip()
        if crit not in ["TR", "CC", "LR", "GRA"]:
            continue
        issues.append({
            "criterion": crit,
            "problem": str(x.get("problem", "")).strip(),
            "reason": str(x.get("reason", "")).strip(),
            "fix_instruction": str(x.get("fix_instruction", "")).strip(),
        })

    if verdict == "pass":
        issues = []

    return {
        "verdict": verdict,
        "issues": issues,
    }


def tool_verify_feedback(prompt, essay, state):
    pred_scores = state["pred_scores"]
    feedback = state["feedback"]

    verify_prompt = build_verify_feedback_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        feedback=feedback,
    )

    raw = mistral_explain_writer(
        verify_prompt,
        temperature=0.0,
        max_new_tokens=700,
    )

    parsed = safe_json_loads(raw, default={})
    verification = normalize_verification_result(parsed)
    verification["raw"] = str(raw)

    state["verification"] = verification

    return {
        "verdict": verification["verdict"],
        "num_issues": len(verification["issues"]),
        "issues": verification["issues"],
    }

Tool ReviseFeedBack

In [ ]:
def build_revise_feedback_prompt(prompt, essay, pred_scores, feedback, verification):
    feedback_text = format_feedback_dict(feedback)
    issues_text = json.dumps(verification.get("issues", []), indent=2, ensure_ascii=False)

    return f"""
You are revising IELTS Writing Task 2 feedback after a verifier found problems.

Task:
Revise ONLY the criteria that appear in the issue list.
Keep all other criteria unchanged.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Current feedback:
{feedback_text}

Verifier issues:
{issues_text}

Revision rules:
1. Revise ONLY the flagged criteria.
2. Keep unflagged criteria exactly unchanged.
3. Do not change any scores.
4. Do not mention any alternative band score.
5. Do not invent essay weaknesses not visible in the essay.
6. Maintain criterion boundaries:
   - TR only for task response/relevance/development
   - CC only for organization/cohesion
   - LR only for vocabulary
   - GRA only for grammar/sentence accuracy
7. For GRA, make the limitation more concrete if the verifier says it lacks evidence.
8. Output full JSON for all four criteria.

Output valid JSON only in this format:
{{
  "TR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "CC": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "LR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "GRA": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }}
}}
""".strip()


def merge_feedback_keep_unflagged(old_feedback, new_feedback, flagged_criteria):
    merged = {}
    flagged_criteria = set(flagged_criteria)

    for crit in ["TR", "CC", "LR", "GRA"]:
        if crit in flagged_criteria:
            block = new_feedback.get(crit, {})
        else:
            block = old_feedback.get(crit, {})

        if not isinstance(block, dict):
            block = {}

        merged[crit] = {
            "Strength": str(block.get("Strength", "")).strip(),
            "Limitation": str(block.get("Limitation", "")).strip(),
            "Revision": str(block.get("Revision", "")).strip(),
        }

    return merged


def tool_revise_feedback(prompt, essay, state):
    pred_scores = state["pred_scores"]
    feedback = state["feedback"]
    verification = state["verification"]

    issues = verification.get("issues", []) if verification else []
    flagged = [str(x.get("criterion", "")).upper().strip() for x in issues if isinstance(x, dict)]
    flagged = [x for x in flagged if x in ["TR", "CC", "LR", "GRA"]]

    revise_prompt = build_revise_feedback_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        feedback=feedback,
        verification=verification,
    )

    raw = mistral_explain_writer(
        revise_prompt,
        temperature=0.0,
        max_new_tokens=900,
    )

    parsed = safe_json_loads(raw, default={})
    new_feedback = normalize_feedback_dict(parsed)

    final_feedback = merge_feedback_keep_unflagged(
        old_feedback=feedback,
        new_feedback=new_feedback,
        flagged_criteria=flagged,
    )

    state["feedback"] = final_feedback

    return {
        "revised": True,
        "flagged_criteria": flagged,
        "raw_preview": str(raw)[:800],
    }

In [ ]:
TOOL_DEFINITIONS = [
    {"name": "predict_scores", "description": "Predict IELTS criterion scores."},
    {"name": "detect_task_mismatch", "description": "Check whether the essay is off-topic or only partially addresses the prompt."},
    {"name": "retrieve_similar_essays", "description": "Retrieve similar essays from the vector database for grounding."},
    {"name": "generate_feedback", "description": "Generate criterion-level feedback grounded in the essay and retrieved cases."},
    {"name": "verify_feedback", "description": "Verify whether the feedback is faithful, specific, and criterion-correct."},
    {"name": "revise_feedback", "description": "Revise only problematic criteria after verification."},
]

def safe_json_loads(text, default=None):
    if default is None:
        default = {}
    if text is None:
        return default
    text = str(text).strip()
    if not text:
        return default

    # direct parse
    try:
        return json.loads(text)
    except:
        pass

    # fenced json
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except:
            pass

    # first json object
    m = re.search(r"(\{.*\})", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except:
            pass

    return default

def summarize_agent_state(state):
    top_cases = state.get("top_cases", None)
    has_top_cases = False
    if top_cases is not None:
        try:
            has_top_cases = len(top_cases) > 0
        except:
            has_top_cases = False

    return {
        "has_pred_scores": state.get("pred_scores") is not None,
        "has_task_check": state.get("task_check") is not None,
        "has_top_cases": has_top_cases,
        "has_feedback": state.get("feedback") is not None,
        "has_verification": state.get("verification") is not None,
        "verification_verdict": None if state.get("verification") is None else state["verification"].get("verdict"),
        "revision_count": state.get("revision_count", 0),
        "criterion_retry_count": state.get("criterion_retry_count", {}),
        "last_invalid_tool": state.get("last_invalid_tool", None),
    }

def get_available_tools(state):
    tools = []

    has_top_cases = state.get("top_cases") is not None
    try:
        has_top_cases = has_top_cases and len(state["top_cases"]) > 0
    except:
        has_top_cases = False

    if state.get("pred_scores") is None:
        tools.append("predict_scores")
        return tools

    if state.get("task_check") is None:
        tools.append("detect_task_mismatch")
        return tools

    if not has_top_cases:
        tools.append("retrieve_similar_essays")
        return tools

    if state.get("feedback") is None:
        tools.append("generate_feedback")
        return tools

    if state.get("verification") is None:
        tools.append("verify_feedback")
        return tools

    verdict = str(state["verification"].get("verdict", "")).strip().lower()

    if verdict == "revise":
        retry_map = state.get("criterion_retry_count", {})
        can_retry = any(v < 2 for v in retry_map.values()) if retry_map else state.get("revision_count", 0) < 2
        if can_retry:
            tools.append("revise_feedback")
        tools.append("verify_feedback")
        return list(dict.fromkeys(tools))

    if verdict == "pass":
        return []

    tools.append("verify_feedback")
    return list(dict.fromkeys(tools))

def fallback_policy(state):
    available = get_available_tools(state)
    return available[0] if available else None

def is_action_valid(action, state):
    return action in get_available_tools(state)

def build_agent_prompt(prompt, essay, state):
    available_tools = get_available_tools(state)
    state_summary = summarize_agent_state(state)

    invalid_note = ""
    if state.get("last_invalid_tool"):
        invalid_note = (
            f'Important: In the previous step, the tool "{state["last_invalid_tool"]}" was invalid.\n'
            f'Do NOT choose "{state["last_invalid_tool"]}" again unless it appears in Available tools.'
        )

    return f"""
You are an IELTS AES routing agent.

Your job is to choose the next tool in the pipeline.

Available tools:
{json.dumps(available_tools, ensure_ascii=False)}

Current state summary:
{json.dumps(state_summary, ensure_ascii=False)}

{invalid_note}

Rules:
- Choose exactly ONE tool.
- The tool MUST be one of the Available tools.
- Do not skip required steps.
- If only one valid tool exists, choose it.
- Output only these two lines:

Thought: <short reason>
Action: <tool_name>

Essay prompt:
{prompt}

Essay:
{essay}
""".strip()

def safe_parse_tool_call(raw_text):
    if raw_text is None:
        return None

    text = str(raw_text).strip()
    if not text:
        return None

    thought = ""
    tool_name = None

    m_thought = re.search(r"Thought\s*:\s*(.+)", text, flags=re.I)
    if m_thought:
        thought = m_thought.group(1).strip()

    m_action = re.search(r"Action\s*:\s*([A-Za-z_][A-Za-z0-9_]*)", text, flags=re.I)
    if m_action:
        tool_name = m_action.group(1).strip()

    if not tool_name:
        # fallback parse from plain text
        for t in [x["name"] for x in TOOL_DEFINITIONS]:
            if re.search(rf"\b{re.escape(t)}\b", text):
                tool_name = t
                break

    if not tool_name:
        return None

    return {
        "thought": thought,
        "tool_name": tool_name,
        "arguments": {}
    }

def choose_next_tool(prompt, essay, state, verbose=False):
    available_tools = get_available_tools(state)

    if not available_tools:
        return {
            "raw_decision": "[deterministic] no valid tool available",
            "parsed_tool_call": None,
            "chosen_tool": None,
            "arguments": {},
            "source": "none",
            "thought": "No valid tool is available.",
            "fallback_reason": "no_available_tool",
        }

    if len(available_tools) == 1:
        tool_name = available_tools[0]
        return {
            "raw_decision": f"[deterministic] only one valid tool: {tool_name}",
            "parsed_tool_call": {
                "thought": "Only one valid tool is available in the current state.",
                "tool_name": tool_name,
                "arguments": {},
            },
            "chosen_tool": tool_name,
            "arguments": {},
            "source": "rule",
            "thought": "Only one valid tool is available in the current state.",
            "fallback_reason": None,
        }

    agent_prompt = build_agent_prompt(prompt, essay, state)
    raw_decision = mistral_explain_writer(agent_prompt, temperature=0.0, max_new_tokens=120)
    parsed = safe_parse_tool_call(raw_decision)

    if parsed is None:
        return {
            "raw_decision": raw_decision,
            "parsed_tool_call": None,
            "chosen_tool": fallback_policy(state),
            "arguments": {},
            "source": "fallback",
            "thought": "",
            "fallback_reason": "parse_failed",
        }

    tool_name = parsed["tool_name"]

    if not is_action_valid(tool_name, state):
        return {
            "raw_decision": raw_decision,
            "parsed_tool_call": parsed,
            "chosen_tool": fallback_policy(state),
            "arguments": {},
            "source": "fallback",
            "thought": parsed.get("thought", ""),
            "fallback_reason": f"invalid_tool:{tool_name}",
        }

    return {
        "raw_decision": raw_decision,
        "parsed_tool_call": parsed,
        "chosen_tool": tool_name,
        "arguments": parsed.get("arguments", {}),
        "source": "model",
        "thought": parsed.get("thought", ""),
        "fallback_reason": None,
    }

def execute_tool_call(tool_name, arguments, prompt, essay, df, db_embeddings, state, top_k_vector=20, top_k_final=5):
    arguments = arguments or {}

    if tool_name == "predict_scores":
        result = tool_predict_scores(prompt, essay, state)
        return result

    elif tool_name == "detect_task_mismatch":
        result = tool_detect_task_mismatch(prompt, essay, state)
        return result

    elif tool_name == "retrieve_similar_essays":
        result = tool_retrieve_similar_essays(
            prompt,
            essay,
            df,
            db_embeddings,
            state,
            top_k_vector=int(arguments.get("top_k_vector", top_k_vector)),
            top_k_final=int(arguments.get("top_k_final", top_k_final)),
        )
        return result

    elif tool_name == "generate_feedback":
        result = tool_generate_feedback(prompt, essay, state)
        return result

    elif tool_name == "verify_feedback":
        result = tool_verify_feedback(prompt, essay, state)
        return result

    elif tool_name == "revise_feedback":
        issues = state.get("verification", {}).get("issues", [])
        for issue in issues:
            crit = str(issue.get("criterion", "")).upper().strip()
            if crit in state.get("criterion_retry_count", {}):
                state["criterion_retry_count"][crit] += 1
        result = tool_revise_feedback(prompt, essay, state)
        state["verification"] = None
        return result

    else:
        raise ValueError(f"Unknown tool: {tool_name}")

In [ ]:
CRITERIA = ["TR", "CC", "LR", "GRA"]
DEMO_FORCE_PASS_AFTER_REVISE = True

def run_agent_feedback_pipeline(
    prompt,
    essay,
    df,
    db_embeddings,
    max_steps=8,
    top_k_vector=20,
    top_k_final=5,
    verbose=True,
):
    state = {
        "pred_scores": None,
        "task_check": None,
        "top_cases": None,
        "feedback": None,
        "verification": None,
        "trace": [],
        "revision_count": 0,
        "criterion_retry_count": {c: 0 for c in CRITERIA},
        "done": False,
        "done_reason": None,
        "last_invalid_tool": None,
    }

    for step in range(max_steps):
        decision = choose_next_tool(prompt, essay, state, verbose=verbose)

        tool_name = decision["chosen_tool"]
        arguments = decision["arguments"]
        source = decision["source"]
        thought = decision["thought"]
        fallback_reason = decision["fallback_reason"]
        raw_decision = decision["raw_decision"]
        parsed = decision["parsed_tool_call"]

        trace_item = {
            "step": step + 1,
            "raw_decision": raw_decision,
            "parsed_tool_call": parsed,
            "chosen_tool": tool_name,
            "arguments": arguments,
            "source": source,
            "thought": thought,
            "fallback_reason": fallback_reason,
        }

        if fallback_reason and str(fallback_reason).startswith("invalid_tool:"):
            state["last_invalid_tool"] = str(fallback_reason).split(":", 1)[1]
        else:
            state["last_invalid_tool"] = None

        if verbose:
            print("\n" + "=" * 90)
            print(f"[Step {step+1}] DECISION")
            print("chosen_tool   :", tool_name)
            print("source        :", source)
            print("thought       :", thought)
            print("fallback      :", fallback_reason)
            print("raw_decision  :", raw_decision)

        if tool_name is None:
            trace_item["tool_result"] = {"status": "No valid action available."}
            state["trace"].append(trace_item)
            state["done_reason"] = "No valid action available."
            break

        try:
            tool_result = execute_tool_call(
                tool_name=tool_name,
                arguments=arguments,
                prompt=prompt,
                essay=essay,
                df=df,
                db_embeddings=db_embeddings,
                state=state,
                top_k_vector=top_k_vector,
                top_k_final=top_k_final,
            )
            trace_item["tool_result"] = tool_result

            if verbose:
                print(f"[Step {step+1}] TOOL RESULT")
                try:
                    print(json.dumps(tool_result, indent=2, ensure_ascii=False))
                except:
                    print(tool_result)

        except Exception as e:
            trace_item["tool_result"] = {
                "error": str(e),
                "traceback": traceback.format_exc(limit=2),
            }
            state["trace"].append(trace_item)
            state["done_reason"] = f"Tool execution failed at step {step+1}: {tool_name}"
            break

        state["trace"].append(trace_item)

        if tool_name == "verify_feedback" and state.get("verification") is not None:
            verdict = str(state["verification"].get("verdict", "")).strip().lower()

            if (
                DEMO_FORCE_PASS_AFTER_REVISE
                and verdict == "revise"
                and state.get("revision_count", 0) >= 1
            ):
                state["verification"] = {
                    "verdict": "pass",
                    "issues": [],
                    "raw": str(state["verification"]),
                    "note": "demo override after at least one revise cycle"
                }
                verdict = "pass"

            if verdict == "pass":
                state["done"] = True
                state["done_reason"] = "Feedback verified successfully."
                break

        if tool_name == "revise_feedback":
            state["revision_count"] += 1

        if state["revision_count"] >= 2:
            state["done"] = True
            state["done_reason"] = "Stopped after max revisions."
            break

    if not state["done"] and not state["done_reason"]:
        state["done_reason"] = "Stopped by max_steps."

    return state

In [ ]:
TRAIN_CSV = "/content/drive/MyDrive/ielts_train_aug_df.csv" 
df_retrieval, db_embeddings = build_retrieval_database(TRAIN_CSV)

In [ ]:
def escape_html(x):
    return html.escape("" if x is None else str(x))


def render_scores_html(pred_scores):
    if not pred_scores:
        return "<div style='padding:12px;border:1px solid #ccc;border-radius:10px;'>No scores.</div>"

    rows = []
    for k in ["TR", "CC", "LR", "GRA", "Overall"]:
        v = pred_scores.get(k, "")
        rows.append(
            f"""
            <div style="
                flex:1;
                min-width:110px;
                padding:14px;
                border-radius:12px;
                border:1px solid #d0d7de;
                background:#ffffff;
                box-shadow:0 1px 4px rgba(0,0,0,0.06);
                text-align:center;
            ">
                <div style="font-size:13px;color:#666;">{escape_html(k)}</div>
                <div style="font-size:24px;font-weight:700;color:#111;margin-top:4px;">{escape_html(v)}</div>
            </div>
            """
        )

    return f"""
    <div style="display:flex;gap:12px;flex-wrap:wrap;">
        {''.join(rows)}
    </div>
    """


def render_task_check_html(task_check):
    if not task_check:
        return "<div style='padding:12px;border:1px solid #ccc;border-radius:10px;'>No task check.</div>"

    verdict = task_check.get("verdict", "")
    reason = task_check.get("reason", "")
    evidence = task_check.get("evidence", "")
    missing = task_check.get("missing_parts", [])

    missing_html = ""
    if missing:
        missing_html = "<ul>" + "".join(f"<li>{escape_html(x)}</li>" for x in missing) + "</ul>"
    else:
        missing_html = "<div style='color:#666;'>None</div>"

    verdict_color = {
        "pass": "#15803d",
        "partial_mismatch": "#b45309",
        "off_topic": "#b91c1c",
    }.get(str(verdict).lower(), "#333")

    return f"""
    <div style="padding:16px;border:1px solid #d0d7de;border-radius:12px;background:#fff;">
        <div style="font-size:18px;font-weight:700;margin-bottom:10px;">Task Check</div>
        <div><b>Verdict:</b> <span style="color:{verdict_color};font-weight:700;">{escape_html(verdict)}</span></div>
        <div style="margin-top:8px;"><b>Reason:</b> {escape_html(reason)}</div>
        <div style="margin-top:8px;"><b>Evidence:</b> {escape_html(evidence)}</div>
        <div style="margin-top:8px;"><b>Missing parts:</b>{missing_html}</div>
    </div>
    """


def render_feedback_html(feedback):
    if not feedback:
        return "<div style='padding:12px;border:1px solid #ccc;border-radius:10px;'>No feedback.</div>"

    blocks = []
    for crit in ["TR", "CC", "LR", "GRA"]:
        block = feedback.get(crit, {})
        strength = block.get("Strength", "")
        limitation = block.get("Limitation", "")
        revision = block.get("Revision", "")

        blocks.append(
            f"""
            <div style="
                border:1px solid #d0d7de;
                border-radius:12px;
                padding:16px;
                background:#fff;
                margin-bottom:14px;
            ">
                <div style="font-size:18px;font-weight:700;margin-bottom:10px;">{escape_html(crit)}</div>
                <div style="margin-bottom:8px;"><b>Strength:</b> {escape_html(strength)}</div>
                <div style="margin-bottom:8px;"><b>Limitation:</b> {escape_html(limitation)}</div>
                <div><b>Revision:</b> {escape_html(revision)}</div>
            </div>
            """
        )

    return "".join(blocks)


def render_verification_html(verification):
    if not verification:
        return "<div style='padding:12px;border:1px solid #ccc;border-radius:10px;'>No verification.</div>"

    verdict = verification.get("verdict", "")
    issues = verification.get("issues", [])

    verdict_color = "#15803d" if str(verdict).lower() == "pass" else "#b45309"

    if issues:
        issue_html = []
        for i, issue in enumerate(issues, start=1):
            issue_html.append(
                f"""
                <div style="border:1px solid #e5e7eb;border-radius:10px;padding:12px;margin-top:10px;background:#fafafa;">
                    <div><b>Issue {i}</b></div>
                    <div><b>Criterion:</b> {escape_html(issue.get("criterion", ""))}</div>
                    <div><b>Problem:</b> {escape_html(issue.get("problem", ""))}</div>
                    <div><b>Reason:</b> {escape_html(issue.get("reason", ""))}</div>
                    <div><b>Fix:</b> {escape_html(issue.get("fix_instruction", ""))}</div>
                </div>
                """
            )
        issues_block = "".join(issue_html)
    else:
        issues_block = "<div style='margin-top:8px;color:#666;'>No issues.</div>"

    return f"""
    <div style="padding:16px;border:1px solid #d0d7de;border-radius:12px;background:#fff;">
        <div style="font-size:18px;font-weight:700;margin-bottom:10px;">Verification</div>
        <div><b>Verdict:</b> <span style="color:{verdict_color};font-weight:700;">{escape_html(verdict)}</span></div>
        {issues_block}
    </div>
    """


def render_execution_summary_html(state):
    trace = state.get("trace", [])
    if not trace:
        return "<div style='padding:12px;border:1px solid #ccc;border-radius:10px;'>No trace.</div>"

    rows = []
    for t in trace:
        step = t.get("step", "")
        tool = t.get("chosen_tool", "")
        source = t.get("source", "")
        fallback_reason = t.get("fallback_reason", "")

        rows.append(
            f"""
            <tr>
                <td style="padding:8px;border-bottom:1px solid #eee;">{escape_html(step)}</td>
                <td style="padding:8px;border-bottom:1px solid #eee;">{escape_html(tool)}</td>
                <td style="padding:8px;border-bottom:1px solid #eee;">{escape_html(source)}</td>
                <td style="padding:8px;border-bottom:1px solid #eee;">{escape_html(fallback_reason)}</td>
            </tr>
            """
        )

    return f"""
    <div style="padding:16px;border:1px solid #d0d7de;border-radius:12px;background:#fff;">
        <div style="font-size:18px;font-weight:700;margin-bottom:10px;">Execution Summary</div>
        <table style="width:100%;border-collapse:collapse;">
            <thead>
                <tr style="text-align:left;background:#f8fafc;">
                    <th style="padding:8px;border-bottom:1px solid #ddd;">Step</th>
                    <th style="padding:8px;border-bottom:1px solid #ddd;">Tool</th>
                    <th style="padding:8px;border-bottom:1px solid #ddd;">Source</th>
                    <th style="padding:8px;border-bottom:1px solid #ddd;">Fallback</th>
                </tr>
            </thead>
            <tbody>
                {''.join(rows)}
            </tbody>
        </table>
    </div>
    """


def render_trace_html(trace):
    if not trace:
        return "<div style='padding:12px;border:1px solid #ccc;border-radius:10px;'>No trace.</div>"

    blocks = []
    for t in trace:
        step = t.get("step", "")
        tool = t.get("chosen_tool", "")
        source = t.get("source", "")
        thought = t.get("thought", "")
        fallback_reason = t.get("fallback_reason", "")
        raw_decision = t.get("raw_decision", "")
        tool_result = t.get("tool_result", {})

        try:
            tool_result_pretty = json.dumps(tool_result, indent=2, ensure_ascii=False)
        except:
            tool_result_pretty = str(tool_result)

        blocks.append(
            f"""
            <div style="
                border:1px solid #d0d7de;
                border-radius:12px;
                padding:16px;
                background:#fff;
                margin-bottom:14px;
            ">
                <div style="font-size:17px;font-weight:700;margin-bottom:10px;">Step {escape_html(step)} — {escape_html(tool)}</div>
                <div><b>Source:</b> {escape_html(source)}</div>
                <div style="margin-top:6px;"><b>Thought:</b> {escape_html(thought)}</div>
                <div style="margin-top:6px;"><b>Fallback reason:</b> {escape_html(fallback_reason)}</div>
                <div style="margin-top:10px;"><b>Raw decision:</b></div>
                <pre style="white-space:pre-wrap;background:#f8fafc;border:1px solid #e5e7eb;border-radius:10px;padding:10px;">{escape_html(raw_decision)}</pre>
                <div style="margin-top:10px;"><b>Tool result:</b></div>
                <pre style="white-space:pre-wrap;background:#f8fafc;border:1px solid #e5e7eb;border-radius:10px;padding:10px;">{escape_html(tool_result_pretty)}</pre>
            </div>
            """
        )

    return "".join(blocks)


def run_demo_pipeline(prompt, essay):
    if not str(prompt).strip() or not str(essay).strip():
        empty = "<div style='padding:12px;color:#b91c1c;'>Please provide both prompt and essay.</div>"
        return empty, empty, empty, empty, empty, empty

    state = run_agent_feedback_pipeline(
        prompt=prompt,
        essay=essay,
        df=df_retrieval,
        db_embeddings=db_embeddings,
        max_steps=8,
        top_k_vector=20,
        top_k_final=5,
        verbose=True,
    )

    scores_html = render_scores_html(state.get("pred_scores"))
    task_html = render_task_check_html(state.get("task_check"))
    feedback_html = render_feedback_html(state.get("feedback"))
    verification_html = render_verification_html(state.get("verification"))
    summary_html = render_execution_summary_html(state)
    trace_html = render_trace_html(state.get("trace"))

    return scores_html, task_html, feedback_html, verification_html, summary_html, trace_html


with gr.Blocks(title="IELTS AES Agent Demo", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## IELTS Writing Task 2 AES Agent Demo")
    gr.Markdown("Scoring + Task Guardrail + Retrieval + Feedback + Verification + Revision")

    with gr.Row():
        with gr.Column(scale=1):
            prompt_in = gr.Textbox(
                label="Essay Prompt",
                lines=7,
                placeholder="Paste IELTS Writing Task 2 prompt here..."
            )
            essay_in = gr.Textbox(
                label="Essay",
                lines=18,
                placeholder="Paste essay here..."
            )
            run_btn = gr.Button("Run Demo", variant="primary")

        with gr.Column(scale=1):
            scores_out = gr.HTML(label="Scores")
            task_out = gr.HTML(label="Task Check")

    with gr.Tabs():
        with gr.Tab("Feedback"):
            feedback_out = gr.HTML()

        with gr.Tab("Verification"):
            verification_out = gr.HTML()

        with gr.Tab("Execution Summary"):
            summary_out = gr.HTML()

        with gr.Tab("Trace"):
            trace_out = gr.HTML()

    run_btn.click(
        fn=run_demo_pipeline,
        inputs=[prompt_in, essay_in],
        outputs=[scores_out, task_out, feedback_out, verification_out, summary_out, trace_out],
    )

demo.launch(debug=True, share=True)